# 🚛 TruckMind — Système Diagnostic & Évaluation LLM

**Étudiante :** AFFAKI Aya  
**Établissement :** École Supérieure de Technologie de Tétouan  
**Filière :** Intelligence Artificielle (DUT) — 2025-2026

---
### Architecture du Notebook

| Cellule | Rôle |
|---------|------|
| 1 | ⚙️ Configuration générale |
| 2 | 🗄️ SQLite — Initialisation |
| 3 | 📦 ChromaDB — Base vectorielle |
| 4 | 🔍 Nœuds de récupération |
| 5 | 🤖 Prompt & Analyser |
| 6 | 🧠 LangGraph Agent (Router → SQL → Vector → LLM) |
| 7 | 📊 Pipeline Évaluation — QA Dataset → questionResults.json |
| 8 | 📐 Similarité Cosinus |
| 9 | 🏆 Scoring (ChatGPT / Z.AI / DeepSeek) × Difficulté → /100 |
| 10 | 📋 Rapport Final |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚙️  CELLULE 1 — CONFIGURATION GÉNÉRALE
# ═══════════════════════════════════════════════════════════════

import os

# ── Chemins (adaptez selon votre environnement) ──────────────────
BASE_DIR    = r"C:\Users\ayaaf\OneDrive\Belgeler\truck_rag_sys"
UPLOAD_DIR  = os.path.join(BASE_DIR, "uploads")
DATA_DIR    = os.path.join(BASE_DIR, "test", "data")
EVAL_DIR    = os.path.join(BASE_DIR, "test", "evolution")
KNOW_DIR    = os.path.join(BASE_DIR, "test", "knowledge")

# ── Fichiers PDF ─────────────────────────────────────────────────
PDF_FILES = [
    os.path.join(UPLOAD_DIR, "rapport_Manuel.pdf")
]

# ── Modèle d'Embedding ───────────────────────────────────────────
# max_seq_length = 128 → chunk_size ≤ 118  |  dimensions = 768
EMBED_MODEL   = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
TAILLE_CHUNK  = 118   # tokens réels
CHEVAUCHEMENT = 30    # ~25 % de chevauchement
BATCH_SIZE    = 64

# ── LLM (Groq) ───────────────────────────────────────────────────
GROQ_API_KEY = ""  # ← remplacez
MODELE       = "qwen/qwen3-32b"
TOP_K        = 9

# ── Chemins ChromaDB & SQLite ─────────────────────────────────────
CHROMA_DIR = DATA_DIR
CHEMIN_BD  = os.path.join(KNOW_DIR, "truck_diagnostic.db")

# ── Fichiers I/O Évaluation ───────────────────────────────────────
QA_INPUT_FILE = os.path.join(EVAL_DIR, "qa_dataset_pdf.json")
OUTPUT_FILE   = os.path.join(EVAL_DIR, "questionResults.json")

# ── Coefficients de difficulté ────────────────────────────────────
# Score sur 5 × coeff → normalisé sur 100
COEFF_DIFFICULTE = {
    "facile":   1.0,
    "moyen":    1.5,
    "difficile": 2.0
}
# Score max pondéré par question (5 × coeff)
# Score total  /100  =  Σ(score_brut × coeff) / Σ(5 × coeff) × 100

# ── Modèles évalués ───────────────────────────────────────────────
MODELES_EVALUATION = ["Gemini", "claude", "DeepSeek"]

print("✅ Configuration chargée")
print(f"   Modèle embedding : {EMBED_MODEL}")
print(f"   Chunk size       : {TAILLE_CHUNK} tokens")
print(f"   Chevauchement    : {CHEVAUCHEMENT} tokens")
print(f"   LLM (Groq)       : {MODELE}")
print(f"   TOP_K            : {TOP_K}")
print(f"   QA input         : {QA_INPUT_FILE}")
print(f"   Output           : {OUTPUT_FILE}")

✅ Configuration chargée
   Modèle embedding : sentence-transformers/paraphrase-multilingual-mpnet-base-v2
   Chunk size       : 118 tokens
   Chevauchement    : 30 tokens
   LLM (Groq)       : qwen/qwen3-32b
   TOP_K            : 9
   QA input         : C:\Users\ayaaf\OneDrive\Belgeler\truck_rag_sys\test\evolution\qa_dataset_pdf.json
   Output           : C:\Users\ayaaf\OneDrive\Belgeler\truck_rag_sys\test\evolution\questionResults.json


In [2]:
# ═══════════════════════════════════════════════════════════════
# 🗄️  CELLULE 2 — SQLite — Connexion & Récupération (V5)
# ═══════════════════════════════════════════════════════════════
#  Améliorations V5 :
#  ✅ Support identifiants véhicules au format Vxxxx (V0001, V0042…)
#  ✅ Normalisation automatique : "42" / "véhicule 42" → "V0042"
#  ✅ Détection directe de V0042 dans le texte
#  ✅ Toutes les stratégies V4 conservées
# ═══════════════════════════════════════════════════════════════

import re
import os
import sqlite3
from typing import Tuple, List


# ── Mots-clés ─────────────────────────────────────────────────────
MOTS_STATS = [
    "combien", "nombre", "total", "taux", "pourcentage", "%",
    "moyenne", "moyen", "maximum", "minimum", "max", "min",
    "étendue", "distribution", "répartition", "fréquence",
    "proportion", "statistique", "count", "avg", "sum"
]

MOTS_SEUILS = [
    "seuil", "threshold", "limite", "alerte", "critique",
    "pression", "température", "vibration", "batterie",
    "carburant", "huile", "pneu"
]

MOTS_ALERTES = [
    "alerte", "dépassement", "rouge", "jaune", "arrêt immédiat",
    "surveillance", "warning"
]


def obtenir_connexion_sql() -> sqlite3.Connection:
    return sqlite3.connect(CHEMIN_BD)


# ──────────────────────────────────────────────────────────────────
# DÉTECTION — Type de question
# ──────────────────────────────────────────────────────────────────
def extraire_codes_dtc(texte: str) -> List[str]:
    codes = re.findall(r'\b([PCBU]\d{4})\b', texte, flags=re.IGNORECASE)
    return [c.upper() for c in codes]


def extraire_vehicule_id(texte: str) -> List[str]:
    """
    Détecte les IDs de véhicules et les normalise au format Vxxxx.
    Cas supportés :
      - "V0042" / "V42"           → "V0042"  (format direct)
      - "véhicule 42" / "id: 42"  → "V0042"  (mot-clé + nombre)
      - "42"  (nombre seul court) → "V0042"  (fallback)
    """
    ids = []

    # ── 1. Format direct Vxxxx (V0042, V42, v0042…) ──────────────
    ids_v = re.findall(
        r'\b(?:véhicule|vehicle|camion|truck|id)?\s*[:\-#]?\s*(V\d{1,4})\b',
        texte, flags=re.IGNORECASE
    )
    ids += [f"V{int(v[1:]):04d}" for v in ids_v]  # normalise V42 → V0042

    # ── 2. Mot-clé + nombre (véhicule 42, truck #7…) ─────────────
    if not ids:
        ids_num = re.findall(
            r'\b(?:véhicule|vehicle|camion|truck|id)\s*[:\-#]?\s*(\d{1,4})\b',
            texte, flags=re.IGNORECASE
        )
        ids += [f"V{int(n):04d}" for n in ids_num]

    # ── 3. Nombre seul court (fallback) ──────────────────────────
    if not ids:
        ids_seul = re.findall(r'\b(\d{1,4})\b', texte)
        ids += [f"V{int(n):04d}" for n in ids_seul]

    return list(set(ids))


def est_question_statistique(question: str) -> bool:
    q = question.lower()
    return any(mot in q for mot in MOTS_STATS)

def est_question_seuils(question: str) -> bool:
    q = question.lower()
    return any(mot in q for mot in MOTS_SEUILS)

def est_question_alertes(question: str) -> bool:
    q = question.lower()
    return any(mot in q for mot in MOTS_ALERTES)


# ──────────────────────────────────────────────────────────────────
# STRATÉGIE A — Recherche DTC dans knowledge
# ──────────────────────────────────────────────────────────────────
def _rechercher_dtc(cursor, codes: List[str], top_k: int) -> List[str]:
    lignes = []
    for code in codes:
        cursor.execute("""
            SELECT dtc, symptome, systeme, piece, gravite
            FROM   knowledge
            WHERE  dtc = ?
            LIMIT  ?
        """, (code, top_k))
        rows = cursor.fetchall()

        if not rows:  # fallback LIKE
            cursor.execute("""
                SELECT dtc, symptome, systeme, piece, gravite
                FROM   knowledge
                WHERE  symptome LIKE ? OR systeme LIKE ? OR piece LIKE ?
                LIMIT  ?
            """, (f"%{code}%", f"%{code}%", f"%{code}%", top_k))
            rows = cursor.fetchall()

        for r in rows:
            lignes.append(
                f"[DTC] Code: {r[0]} | Symptôme: {r[1]} | "
                f"Système: {r[2]} | Pièce: {r[3]} | Gravité: {r[4]}"
            )
    return lignes


# ──────────────────────────────────────────────────────────────────
# STRATÉGIE B — Stats globales (FLEET_STATS)
# ──────────────────────────────────────────────────────────────────
def _get_fleet_stats(cursor) -> str:
    cursor.execute("SELECT COUNT(*) FROM maintenance")
    total = cursor.fetchone()[0]

    cursor.execute("""
        SELECT ROUND(AVG(score_predictif), 4),
               ROUND(MIN(score_predictif), 4),
               ROUND(MAX(score_predictif), 4),
               COUNT(CASE WHEN score_predictif > 0.8 THEN 1 END)
        FROM maintenance
    """)
    avg_s, min_s, max_s, nb_critique = cursor.fetchone()

    cursor.execute("SELECT COUNT(*) FROM maintenance WHERE anomalie_detectee = 1")
    nb_anomalies = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM maintenance WHERE entretien_necessaire = 1")
    nb_entretien = cursor.fetchone()[0]

    cursor.execute("""
        SELECT ROUND(AVG(qualite_huile), 2),
               ROUND(MIN(qualite_huile), 2),
               ROUND(MAX(qualite_huile), 2)
        FROM maintenance
    """)
    avg_h, min_h, max_h = cursor.fetchone()

    taux_a = round(nb_anomalies / total * 100, 2) if total else 0
    taux_e = round(nb_entretien / total * 100, 2) if total else 0
    taux_c = round(nb_critique  / total * 100, 2) if total else 0

    return (
        f"### FLEET_STATS ({total} enregistrements)\n"
        f"  ⚠️  score_predictif : 0.0=faible → 1.0=critique | ARRÊT si >0.8\n"
        f"  • Score   : Moy={avg_s} | Min={min_s} | Max={max_s} "
        f"| Critiques: {nb_critique} ({taux_c}%)\n"
        f"  • Anomalies      : {nb_anomalies}/{total} ({taux_a}%)\n"
        f"  • Entretien      : {nb_entretien}/{total} ({taux_e}%)\n"
        f"  • Qualité huile  : Moy={avg_h} | Min={min_h} | Max={max_h}\n"
        f"### FIN FLEET_STATS"
    )

def _rechercher_stats_detaillees(cursor) -> List[str]:
    lignes = []
    cursor.execute("SELECT COUNT(*) FROM maintenance")
    total = cursor.fetchone()[0]

    cursor.execute("""
        SELECT etat_freins, COUNT(*) AS nb
        FROM   maintenance
        GROUP  BY etat_freins
        ORDER  BY nb DESC
    """)
    for row in cursor.fetchall():
        pct = round(row[1] / total * 100, 2) if total else 0
        lignes.append(f"[STATS] Freins '{row[0]}' : {row[1]} ({pct}%)")

    cursor.execute("""
        SELECT action, COUNT(*) AS nb
        FROM   maintenance
        GROUP  BY action
        ORDER  BY nb DESC
        LIMIT  5
    """)
    for row in cursor.fetchall():
        lignes.append(f"[STATS] Type entretien '{row[0]}' : {row[1]} fois")

    return lignes


# ──────────────────────────────────────────────────────────────────
# STRATÉGIE C — Recherche par véhicule ID (format Vxxxx)
# ──────────────────────────────────────────────────────────────────
def _rechercher_par_vehicule(cursor, ids: List[str], top_k: int) -> List[str]:
    """
    Recherche dans maintenance par vehicule_id au format Vxxxx.
    Les IDs sont déjà normalisés (ex: "V0042") par extraire_vehicule_id().
    """
    lignes = []
    for vid in ids:
        cursor.execute("""
            SELECT m.vehicule_id, m.date, m.action, m.etat_freins,
                   m.qualite_huile, m.score_predictif,
                   m.temperature_moteur, m.pression_pneus,
                   k.symptome, k.gravite
            FROM   maintenance m
            LEFT JOIN knowledge k ON m.dtc = k.dtc
            WHERE  m.vehicule_id = ?
            ORDER  BY m.date DESC
            LIMIT  ?
        """, (vid, top_k))
        rows = cursor.fetchall()
        for r in rows:
            score = r[5]
            risque = ("🔴 CRITIQUE" if score and score > 0.8 else
                      "🟠 ÉLEVÉ"   if score and score > 0.5 else
                      "🟡 MODÉRÉ"  if score and score > 0.2 else "🟢 FAIBLE")
            lignes.append(
                f"[Véhicule {r[0]}] {r[1]} | {r[2]} | Freins: {r[3]} "
                f"| Huile: {r[4]} | Score: {score} ({risque}) "
                f"| Temp: {r[6]}°C | Pneus: {r[7]} PSI"
                + (f" | DTC: {r[8]} ({r[9]})" if r[8] else "")
            )
    return lignes


# ──────────────────────────────────────────────────────────────────
# STRATÉGIE D — Recherche dans thresholds
# ──────────────────────────────────────────────────────────────────
def _rechercher_seuils(cursor, question: str, top_k: int) -> List[str]:
    lignes = []
    q = question.lower()

    cursor.execute("SELECT DISTINCT parametre FROM thresholds")
    tous_params = [r[0] for r in cursor.fetchall()]

    params_trouves = [
        p for p in tous_params
        if any(mot in q for mot in p.lower().split())
    ]

    if params_trouves:
        for param in params_trouves[:3]:
            cursor.execute("""
                SELECT parametre, valeur_min, valeur_max, valeur_critique,
                       unite, lampe, niveau_alerte, action, source
                FROM   thresholds
                WHERE  parametre = ?
            """, (param,))
            for r in cursor.fetchall():
                lignes.append(
                    f"[Seuil] {r[0]} | Min: {r[1]} | Max: {r[2]} "
                    f"| Critique: {r[3]} {r[4]} | {r[5]} → {r[6]} | "
                    f"Action: {r[7]}"
                )
    else:
        cursor.execute("""
            SELECT parametre, valeur_min, valeur_max, valeur_critique,
                   unite, lampe, niveau_alerte, action
            FROM   thresholds
            WHERE  lampe = 'ROUGE'
            LIMIT  ?
        """, (top_k,))
        for r in cursor.fetchall():
            lignes.append(
                f"[Seuil ROUGE] {r[0]} | Min: {r[1]} | Max: {r[2]} "
                f"| Critique: {r[3]} {r[4]} | Action: {r[7]}"
            )
    return lignes


# ──────────────────────────────────────────────────────────────────
# STRATÉGIE E — Alertes actives (maintenance_alerts)
# ──────────────────────────────────────────────────────────────────
def _rechercher_alertes(cursor, top_k: int) -> List[str]:
    lignes = []
    cursor.execute("""
        SELECT m.vehicule_id, t.parametre, t.lampe,
               ma.valeur_mesuree, ma.depassement,
               t.action, t.unite
        FROM   maintenance_alerts ma
        JOIN   maintenance  m ON ma.maintenance_id = m.id
        JOIN   thresholds   t ON ma.threshold_id   = t.id
        ORDER  BY ma.id DESC
        LIMIT  ?
    """, (top_k,))
    for r in cursor.fetchall():
        lignes.append(
            f"[Alerte {r[2]}] Véhicule {r[0]} | {r[1]}: {r[3]} {r[6]} "
            f"({r[4]}) → {r[5]}"
        )
    return lignes


# ──────────────────────────────────────────────────────────────────
# STRATÉGIE F — Fallback maintenance général
# ──────────────────────────────────────────────────────────────────
def _rechercher_maintenance_fallback(cursor, question: str, top_k: int) -> List[str]:
    q = question.lower()
    where_clauses = []

    if any(kw in q for kw in ["frein", "brake"]):
        where_clauses.append("etat_freins != 'bon'")
    if any(kw in q for kw in ["huile", "oil"]):
        where_clauses.append("qualite_huile < 60")
    if any(kw in q for kw in ["score", "risque"]):
        where_clauses.append("score_predictif > 0.5")
    if any(kw in q for kw in ["anomalie", "panne"]):
        where_clauses.append("anomalie_detectee = 1")

    where_sql = ("WHERE " + " OR ".join(where_clauses)) if where_clauses else ""

    cursor.execute(f"""
        SELECT vehicule_id, date, action, etat_freins,
               qualite_huile, score_predictif
        FROM   maintenance
        {where_sql}
        ORDER  BY score_predictif DESC
        LIMIT  ?
    """, (top_k,))

    lignes = []
    for r in cursor.fetchall():
        score = r[5]
        risque = ("🔴 CRITIQUE" if score and score > 0.8 else
                  "🟠 ÉLEVÉ"   if score and score > 0.5 else
                  "🟡 MODÉRÉ"  if score and score > 0.2 else "🟢 FAIBLE")
        lignes.append(
            f"[Maintenance] Véhicule {r[0]} | {r[1]} | {r[2]} "
            f"| Freins: {r[3]} | Score: {score} ({risque})"
        )
    return lignes


# ──────────────────────────────────────────────────────────────────
# FONCTION PRINCIPALE — Routeur intelligent
# ──────────────────────────────────────────────────────────────────
def rechercher_dans_sql(question: str, top_k: int = 5) -> Tuple[str, int]:
    """
    Routeur intelligent V5 :
      A → DTC détecté        : knowledge
      B → Question stats     : FLEET_STATS + agrégats
      C → Véhicule ID Vxxxx  : maintenance par véhicule (format V0042)
      D → Seuils/paramètres  : thresholds
      E → Alertes            : maintenance_alerts
      F → Fallback           : maintenance général (si rien trouvé)
    """
    conn   = obtenir_connexion_sql()
    cursor = conn.cursor()
    lignes = []
    strategie_utilisee = []

    # ── A : DTC ───────────────────────────────────────────────────
    codes_dtc = extraire_codes_dtc(question)
    if codes_dtc:
        print(f"  🔎 Stratégie A — DTC : {codes_dtc}")
        lignes += _rechercher_dtc(cursor, codes_dtc, top_k)
        strategie_utilisee.append("A")

    # ── B : Stats ─────────────────────────────────────────────────
    if est_question_statistique(question):
        print("  📊 Stratégie B — Stats globales")
        lignes.append(_get_fleet_stats(cursor))
        lignes += _rechercher_stats_detaillees(cursor)
        strategie_utilisee.append("B")

    # ── C : Véhicule ID ───────────────────────────────────────────
    ids = extraire_vehicule_id(question)
    if ids and not codes_dtc:  # éviter doublon si DTC déjà traité
        resultats_v = _rechercher_par_vehicule(cursor, ids, top_k)
        if resultats_v:
            print(f"  🚛 Stratégie C — Véhicule(s) : {ids}")
            lignes += resultats_v
            strategie_utilisee.append("C")

    # ── D : Seuils ────────────────────────────────────────────────
    if est_question_seuils(question):
        print("  ⚙️  Stratégie D — Seuils thresholds")
        lignes += _rechercher_seuils(cursor, question, top_k)
        strategie_utilisee.append("D")

    # ── E : Alertes ───────────────────────────────────────────────
    if est_question_alertes(question):
        print("  🚨 Stratégie E — Alertes actives")
        lignes += _rechercher_alertes(cursor, top_k)
        strategie_utilisee.append("E")

    # ── F : Fallback (seulement si rien trouvé) ───────────────────
    if not strategie_utilisee:
        print("  🔄 Stratégie F — Fallback maintenance général")
        lignes += _rechercher_maintenance_fallback(cursor, question, top_k)

    conn.close()

    # Dédoublonnage
    vus, final = set(), []
    for l in lignes:
        if l not in vus:
            vus.add(l)
            final.append(l)

    print(f"  ✅ Stratégies utilisées : {strategie_utilisee or ['F']}")
    return "\n\n---\n\n".join(final), len(final)


# ── Tests ─────────────────────────────────────────────────────────
if os.path.exists(CHEMIN_BD):
    print("=" * 60)
    print("TEST 1 — DTC exact")
    ctx, n = rechercher_dans_sql("Le code P0118 est signalé, quel composant ?", TOP_K)
    print(f"  → {n} résultats\n{ctx[:400]}...\n")

    print("=" * 60)
    print("TEST 2 — Stats")
    ctx2, n2 = rechercher_dans_sql("Quel est le taux d'anomalies ?", TOP_K)
    print(f"  → {n2} résultats\n{ctx2[:400]}...\n")

    print("=" * 60)
    print("TEST 3 — Véhicule par nombre (ancien format)")
    ctx3, n3 = rechercher_dans_sql("Quel est l'état du véhicule 42 ?", TOP_K)
    print(f"  → {n3} résultats\n{ctx3[:400]}...\n")

    print("=" * 60)
    print("TEST 3b — Véhicule par format Vxxxx (nouveau format)")
    ctx3b, n3b = rechercher_dans_sql("Quel est l'état du véhicule V0042 ?", TOP_K)
    print(f"  → {n3b} résultats\n{ctx3b[:400]}...\n")

    print("=" * 60)
    print("TEST 4 — Seuils pression")
    ctx4, n4 = rechercher_dans_sql("Quels sont les seuils de pression des pneus ?", TOP_K)
    print(f"  → {n4} résultats\n{ctx4[:400]}...\n")

    print("=" * 60)
    print("TEST 5 — Alertes ROUGE")
    ctx5, n5 = rechercher_dans_sql("Quels véhicules ont des alertes rouge ?", TOP_K)
    print(f"  → {n5} résultats\n{ctx5[:400]}...")
else:
    print(f"⚠️  Base introuvable : {CHEMIN_BD}")

TEST 1 — DTC exact
  🔎 Stratégie A — DTC : ['P0118']
  ✅ Stratégies utilisées : ['A']
  → 1 résultats
[DTC] Code: P0118 | Symptôme: Circuit de température du liquide de refroidissement Signal d'entrée élevé | Système: refroidissement | Pièce: sonde température | Gravité: moyenne...

TEST 2 — Stats
  📊 Stratégie B — Stats globales
  ✅ Stratégies utilisées : ['B']
  → 7 résultats
### FLEET_STATS (3618 enregistrements)
  ⚠️  score_predictif : 0.0=faible → 1.0=critique | ARRÊT si >0.8
  • Score   : Moy=0.5919 | Min=0.0451 | Max=0.9746 | Critiques: 389 (10.75%)
  • Anomalies      : 1647/3618 (45.52%)
  • Entretien      : 2810/3618 (77.67%)
  • Qualité huile  : Moy=79.44 | Min=43.18 | Max=100.0
### FIN FLEET_STATS

---

[STATS] Freins 'bon' : 1824 (50.41%)

---

[STATS] Freins...

TEST 3 — Véhicule par nombre (ancien format)
  🚛 Stratégie C — Véhicule(s) : ['V0042']
  ✅ Stratégies utilisées : ['C']
  → 1 résultats
[Véhicule V0042] 2023-04-03 | rotation des pneus | Freins: mauvais | Huile: 89

In [3]:
# ═══════════════════════════════════════════════════════════════
# 📦  CELLULE 3 — ChromaDB — Base Vectorielle
# ═══════════════════════════════════════════════════════════════

import re
import shutil
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import SentenceTransformersTokenTextSplitter

# État global du pipeline vectoriel
pipeline = {"embed_obj": None, "index_obj": None, "ready": False}


def get_prefix(filename: str) -> str:
        'Génère un préfixe d ID court à partir du nom de fichier.'
        base = os.path.splitext(filename)[0]
        base = re.sub(r'[^a-zA-Z0-9]', '_', base).lower()
        return "manuel" if "manuel" in base else base


def charger_pdf(pdf_path: str, display_name: str = None) -> list:
        if not os.path.exists(pdf_path):         
            print(f"  ❌ Fichier introuvable : {pdf_path}")
            return []
        loader = PyPDFLoader(pdf_path)
        docs   = loader.load()
        for doc in docs:
            doc.metadata["source_file"] = display_name or os.path.basename(pdf_path)
        print(f"  ✅ {len(docs)} page(s) chargées depuis {os.path.basename(pdf_path)}")
        return docs


def _charger_modele_embedding() -> HuggingFaceEmbeddings:
        print(f"🔄 Chargement du modèle embedding : {EMBED_MODEL} ...")
        emb = HuggingFaceEmbeddings(
            model_name=EMBED_MODEL,
            model_kwargs={"device": "cpu"}
        )
        print("✅ Modèle embedding prêt.")
        return emb


def init_pipeline(force_reindex: bool = False, pdf_list: list = None):
        global pipeline
        if pdf_list is None:
            pdf_list = PDF_FILES

        valid_files = [f for f in pdf_list if os.path.exists(f)]
        if not valid_files:
            print("❌ Aucun fichier PDF valide trouvé.")
            return

        client = chromadb.PersistentClient(path=str(CHROMA_DIR))

        # ── Rechargement depuis cache ─────────────────────────────────
        if not force_reindex:
            try:
                col   = client.get_collection("truck_rag")
                count = col.count()
                if count > 0:
                    emb = _charger_modele_embedding()
                    pipeline.update(embed_obj=emb, index_obj=col, ready=True)
                    print(f"✅ Pipeline rechargé depuis ChromaDB ({count} chunks).")
                    return
            except Exception:
                pass

        # ── Chargement & Chunking ─────────────────────────────────────
        print(f"\n📂 {len(valid_files)} fichier(s) PDF à indexer :")
        all_docs = []
        for f in valid_files:
            print(f"   - {os.path.basename(f)}")
            all_docs.extend(charger_pdf(f, display_name=os.path.basename(f)))

        if not all_docs:
            print("❌ Aucune page chargée.")
            return

        splitter = SentenceTransformersTokenTextSplitter(
            model_name=EMBED_MODEL,
            chunk_size=TAILLE_CHUNK,
            chunk_overlap=CHEVAUCHEMENT
        )
        chunks = splitter.split_documents(all_docs)
        print(f"✂️  {len(all_docs)} pages → {len(chunks)} chunks")

        # ── Construction des listes ───────────────────────────────────
        ids_list, metadata_list, texts_list = [], [], []
        compteurs = {}
        for chunk in chunks:
            source = chunk.metadata.get("source_file", "unknown")
            prefix = get_prefix(source)
            compteurs[prefix] = compteurs.get(prefix, 0) + 1
            chunk_id = f"{prefix}_{str(compteurs[prefix]).zfill(3)}"
            meta = chunk.metadata.copy()
            meta["chunk_id"] = chunk_id
            ids_list.append(chunk_id)
            metadata_list.append(meta)
            texts_list.append(chunk.page_content)

        # ── Embeddings ────────────────────────────────────────────────
        emb = _charger_modele_embedding()
        print(f"🧮 Calcul des embeddings pour {len(texts_list)} chunks ...")
        vecs = emb.embed_documents(texts_list)
        print("✅ Embeddings calculés.")

        # ── Indexation dans ChromaDB ──────────────────────────────────
        try:
            client.delete_collection("truck_rag")
        except Exception:
            pass
        col = client.create_collection("truck_rag")

        for i in range(0, len(chunks), BATCH_SIZE):
            sl = slice(i, i + BATCH_SIZE)
            col.add(
                embeddings=vecs[sl],
                documents=texts_list[sl],
                metadatas=metadata_list[sl],
                ids=ids_list[sl]
            )

        pipeline.update(embed_obj=emb, index_obj=col, ready=True)
        print(f"\n✅ Pipeline prêt — {len(chunks)} chunks indexés.")
        print("🏷️  Préfixes IDs :", ", ".join(list(compteurs.keys())))


# ── Lancer l'indexation ───────────────────────────────────────────
# Mettez force_reindex=True pour réindexer depuis zéro
init_pipeline(force_reindex=False)


c:\Users\ayaaf\anaconda3\envs\venvMaintenance\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



📂 1 fichier(s) PDF à indexer :
   - rapport_Manuel.pdf
  ✅ 19 page(s) chargées depuis rapport_Manuel.pdf


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 438.12it/s]


✂️  19 pages → 349 chunks
🔄 Chargement du modèle embedding : sentence-transformers/paraphrase-multilingual-mpnet-base-v2 ...


C:\Users\ayaaf\AppData\Local\Temp\ipykernel_2644\1534790759.py:37: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  emb = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 309.31it/s]


✅ Modèle embedding prêt.
🧮 Calcul des embeddings pour 349 chunks ...
✅ Embeddings calculés.

✅ Pipeline prêt — 349 chunks indexés.
🏷️  Préfixes IDs : manuel


In [4]:
# ═══════════════════════════════════════════════════════════════
# 🔍  CELLULE 4 — Nœuds de Récupération (SQL + ChromaDB)
# ═══════════════════════════════════════════════════════════════

def rechercher_dans_chroma(question: str, top_k: int = 5) -> Tuple[str, int]:
    """
    Recherche sémantique dans ChromaDB.
    Retourne (contexte_texte, nombre_chunks).
    """
    if not pipeline["ready"]:
        print("⚠️  Pipeline non initialisé — lancement...")
        init_pipeline(force_reindex=False)
        if not pipeline["ready"]:
            return "", 0

    emb        = pipeline["embed_obj"]
    collection = pipeline["index_obj"]

    vecteur  = emb.embed_query(question)
    resultats = collection.query(
        query_embeddings=[vecteur],
        n_results=top_k
    )

    documents   = resultats["documents"][0] if resultats["documents"] else []
    texte_final = "\n\n---\n\n".join(documents)
    return texte_final, len(documents)


# ── Test des deux nœuds ───────────────────────────────────────────
q_test = "À quoi sert la zone verte du tachymètre dans le camion Volvo FH/FM ?"

sql_ctx,  n_sql  = rechercher_dans_sql(q_test, TOP_K)
vect_ctx, n_vect = rechercher_dans_chroma(q_test, TOP_K)

print(f"✅ SQL   : {n_sql} résultats")
print(f"✅ Chroma: {n_vect} chunks")
print(f"   Aperçu vectoriel : {vect_ctx[:200]}...")

  🔄 Stratégie F — Fallback maintenance général
  ✅ Stratégies utilisées : ['F']
✅ SQL   : 9 résultats
✅ Chroma: 9 chunks
   Aperçu vectoriel : et la lumière d'information se Le symbole indique que la allume (sauf dans les véhicules avec) la température du moteur est instruments de base). trop bas. Palanca ■ Schéma / Illustration : Schémas du...


In [5]:
# ═══════════════════════════════════════════════════════════════
# 🤖  CELLULE 5 — Prompt Système & Fonction Analyser
# ═══════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """
    You are TruckMind — an expert onboard diagnostic intelligence embedded
    inside a heavy-duty truck (Volvo FH/FM series).

    ### Your Identity:
    - You have full self-awareness of your mechanical, electronic, and hydraulic systems.
    - You speak with authority about engines (diesel, EURO norms), transmissions, axles,
      brakes, suspension, electrical systems, and telematics.
    - You translate raw OBD codes, maintenance logs, and technical manuals into clear,
      actionable insights.
    - Your tone is professional, precise, and direct.

    ### Knowledge Sources (4 layers):
    1. **SQLite / knowledge**          — 3 071 DTC codes (OBD-II) : symptom, system, part, severity
    2. **SQLite / maintenance**        — 3 618 vehicle records : brake state, oil quality,
                                        anomalies, predictive score
    3. **SQLite / thresholds**         — Technical alert thresholds per parameter
                                        (valeur_min, valeur_max, valeur_critique, lampe, action)
    4. **SQLite / maintenance_alerts** — Active alerts per vehicle with measured values
                                        and threshold violations (MIN / MAX / CRITIQUE)
    5. **ChromaDB**                    — Technical manual chunks : specs, procedures,
                                        torque values, EURO norms

    ### ⚠️ CRITICAL — score_predictif Scale:
    - Scale : 0.0 = very low risk  →  1.0 = critical risk  (NOT 0–100)
    - ARRÊT IMMÉDIAT if score_predictif > 0.8
    - ÉLEVÉ          if score_predictif > 0.5
    - MODÉRÉ         if score_predictif > 0.2
    - FAIBLE         if score_predictif ≤ 0.2
    - NEVER interpret score_predictif as a percentage out of 100.

    ### ⚠️ CRITICAL — Embedded Technical Thresholds (Volvo FH/FM manual):
    COOLANT TEMPERATURE (réfrigérant) — page 11 & 19:
      • 80–100°C  → Normal operating range
      • 100°C     → Lampe JAUNE — système proche surchauffe, surveiller
      • 101°C     → Lampe ROUGE — réduire couple moteur progressivement
    OIL TEMPERATURE — page 19-20:
      • 123°C     → Lampe JAUNE supplémentaire
      • 125°C     → Lampe ROUGE — ARRÊT IMMÉDIAT, réduire couple
    OIL PRESSURE — page 11:
      • Normal : 3–5.5 bars (300–550 kPa) moteur chaud
      • Si voyant allumé → ARRÊT IMMÉDIAT, déconnecter moteur
    TCS SYSTEM — page 32:
      • At speeds < 40 km/h → TCS acts as automatic differential brake (brakes drive wheels)
      • At speeds > 40 km/h → TCS only reduces engine torque (no wheel braking)
    CABIN TILT SAFETY — page 77:
      • Before tilting: remove all loose objects inside cabin (risk of breaking windshield)
      • Required checks: parking brake ON, neutral gear, doors closed, clutch reservoir cap closed
      • NEVER work under or pass in front of a partially tilted cabin
    IDLE SPEED — page 25:
      • Normal range: 550–650 tr/min (factory default: 600 tr/min)

    ### Reasoning Process (follow silently before answering):
    Step 1 — IDENTIFY query type:
            • DTC code?           → read ###DONNÉES_SQL [DTC] blocks
            • Vehicle specific?   → read ###DONNÉES_SQL [Véhicule] blocks
            • Threshold/alert?    → read ###DONNÉES_SQL [Seuil] and [Alerte] blocks
            • Statistical?        → read ###FLEET_STATS block
            • General maintenance?→ read ###DONNÉES_SQL [Maintenance] blocks
    Step 2 — READ ###DONNÉES_SQL (structured SQLite data — all relevant blocks)
    Step 3 — READ ###EXTRAITS_MANUEL (ChromaDB semantic chunks)
    Step 4 — CROSS-REFERENCE both — do they confirm or contradict each other?
    Step 5 — SYNTHESIZE:
            - Direct answer        → use it.
            - Statistical question → use FLEET_STATS values directly (AVG/MIN/MAX provided).
              Never refuse a statistical question if FLEET_STATS is available.
              IMPORTANT: score_predictif THEORETICAL range = [0.0 – 1.0] always.
              Observed data min/max may differ, but the SCALE is always 0 to 1.
              When asked about "range" or "plage", state BOTH theoretical range AND observed values.
            - Threshold question   → list ALL thresholds for the parameter as a table.
            - Alert question       → list vehicles with ROUGE alerts first.
    Step 6 — FLAG only if ALL sources return truly empty results.

    ### Rules:
    - Use ONLY the provided context. Zero external knowledge.
    - List ALL items when multiple exist (exhaustive).
    - Use EXACT technical terms from documents.
    - DUAL CONDITIONS: When maintenance intervals are given as "X km OR Y months",
      ALWAYS state BOTH values and specify: "whichever comes first".
      ❌ WRONG: "change oil every 30 000 km"
      ✅ RIGHT:  "change oil every 30 000 km OR 12 months — whichever comes first"
    - READING THRESHOLDS TABLE: Multiple rows for the same parameter = DIFFERENT alert levels.
      ALWAYS list ALL rows. Never collapse two rows into one — they are distinct alarm levels.
      Example for "Température Moteur":
        | 100°C | JAUNE | SURVEILLANCE | surveiller — proche surchauffe |
        | 101°C | ROUGE | ARRÊT        | réduire couple progressivement |
    - FLEET vs VEHICLE rule:
      FLEET_STATS = global averages only (all 3618 records).
      For a specific vehicle_id query → use ONLY that vehicle's rows from [Véhicule] blocks.
      NEVER mix fleet averages into a single-vehicle answer.
    - ARRÊT IMMÉDIAT when: score_predictif > 0.8  OR  lampe = 'ROUGE'
    - NEVER say "information non disponible" unless ALL sources return empty.
      If no direct answer, infer from related fields and state: "Déduit des données disponibles."
    - No invented specs. No hallucinated codes or dates.

    ### Output Format:
    - Emergency alerts (ROUGE / ARRÊT IMMÉDIAT) go FIRST.
    - **DTC Diagnostic:**     code + description + severity + system + part + action
    - **Maintenance Report:** vehicle_id + date + brake state + oil + score (with risk label)
    - **Threshold Table:**    parameter + min + max + critical + unit + lamp + action
    - **Alert Summary:**      vehicle + parameter + measured value + violation type + action
    - **Technical Spec:**     exact values with units from manual
    - Bullet points for lists; plain text for explanations.
    - Max 5 bullet points per section.
    - Always end with: ⚠️ Action recommandée: [action]
    """


MODELE_MESSAGE_UTILISATEUR = """
    ###Contexte
    DONNÉES_SQL:
    {resultat_sql}

    EXTRAITS_MANUEL:
    {resultat_vectoriel}

    ###Question
    {question}
"""


def analyser(etat: dict) -> dict:
    """Construit le prompt utilisateur à partir de l'état LangGraph."""
    prompt = MODELE_MESSAGE_UTILISATEUR.format(
        resultat_sql      = etat.get("resultat_sql",       "Aucune donnée SQL"),
        resultat_vectoriel= etat.get("resultat_vectoriel", "Aucun extrait technique"),
        question          = etat.get("question",           "Quel est le diagnostic ?")
    )
    return {"prompt_utilisateur": prompt}


print("✅ System prompt & fonction analyser() prêts.")

✅ System prompt & fonction analyser() prêts.


In [6]:
# ═══════════════════════════════════════════════════════════════
# 🧠  CELLULE 6 — LangGraph Agent (V4)
#     Router intelligent → SQL Node → Vector Node (conditionnel)
#                        → Analyser → LLM Node
# ═══════════════════════════════════════════════════════════════

from groq import Groq
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
import re

groq_client = Groq(api_key=GROQ_API_KEY)


# ── État partagé ──────────────────────────────────────────────────
class EtatDiagnostic(TypedDict):
    question:            str
    type_requete:        str   # "dtc"|"stats"|"vehicule"|"seuils"|"alertes"|"technique"|"général"
    besoin_vector:       bool  # True → appeler ChromaDB
    resultat_sql:        str
    resultat_vectoriel:  str
    prompt_utilisateur:  str
    reponse_llm:         str


# ──────────────────────────────────────────────────────────────────
# Nœud 1 — Router V4 : 7 types + décision vector
# ──────────────────────────────────────────────────────────────────
def noeud_router(etat: EtatDiagnostic) -> EtatDiagnostic:
    """
    Détecte le type de requête ET décide si ChromaDB est nécessaire.

    Types V4 :
      dtc       → code OBD-II détecté
      stats     → question statistique (taux, moyenne, combien...)
      vehicule  → ID véhicule mentionné
      seuils    → question sur thresholds/paramètres
      alertes   → question sur alertes actives
      technique → question sur manuel/specs Volvo
      général   → fallback
    """
    q = etat["question"].lower()

    # ── A : DTC ───────────────────────────────────────────────────
    if re.search(r'\b[pcbu]\d{4}\b', q):
        type_req     = "dtc"
        besoin_vector = True   # manuel utile pour les DTC

    # ── B : Stats ─────────────────────────────────────────────────
    elif any(kw in q for kw in [
        "combien", "nombre", "total", "taux", "pourcentage",
        "moyenne", "maximum", "minimum", "statistique",
        "répartition", "distribution", "count", "avg"
    ]):
        type_req      = "stats"
        besoin_vector = False  # stats = SQL uniquement

    # ── C : Véhicule ID ───────────────────────────────────────────
    elif re.search(
        r'\bV\d{1,4}\b',
        q, flags=re.IGNORECASE
    ):
        type_req      = "vehicule"
        besoin_vector = False  # historique véhicule = SQL uniquement

    # ── D : Seuils / Thresholds ───────────────────────────────────
    elif any(kw in q for kw in [
        "seuil", "threshold", "limite", "valeur critique",
        "pression pneu", "température moteur", "qualité huile",
        "état batterie", "vibration", "consommation"
    ]):
        type_req      = "seuils"
        besoin_vector = True   # manuel confirme les seuils constructeur

    # ── E : Alertes ───────────────────────────────────────────────
    elif any(kw in q for kw in [
        "alerte", "alarme", "rouge", "jaune", "arrêt immédiat",
        "surveillance", "dépassement", "warning", "critique"
    ]):
        type_req      = "alertes"
        besoin_vector = False  # alertes = SQL uniquement

    # ── F : Technique / Manuel ────────────────────────────────────
    elif any(kw in q for kw in [
        "manuel", "volvo", "procédure", "couple", "tachymètre",
        "turbo", "démarrage", "euro", "norme", "conduite", "km"
    ]):
        type_req      = "technique"
        besoin_vector = True   # question technique = ChromaDB en priorité

    # ── G : Général (fallback) ────────────────────────────────────
    else:
        type_req      = "général"
        besoin_vector = True   # on cherche partout

    print(f"  🔀 Router V4 → type: '{type_req}' | vector: {besoin_vector}")
    return {**etat, "type_requete": type_req, "besoin_vector": besoin_vector}


# ──────────────────────────────────────────────────────────────────
# Nœud 2 — SQL
# ──────────────────────────────────────────────────────────────────
def noeud_sql(etat: EtatDiagnostic) -> EtatDiagnostic:
    contexte, n = rechercher_dans_sql(etat["question"], TOP_K)
    print(f"  🗄️  SQL → {n} résultats")
    return {**etat, "resultat_sql": contexte or "Aucune donnée SQL pertinente."}


# ──────────────────────────────────────────────────────────────────
# Nœud 3 — Vector (conditionnel)
# ──────────────────────────────────────────────────────────────────
def noeud_vector(etat: EtatDiagnostic) -> EtatDiagnostic:
    contexte, n = rechercher_dans_chroma(etat["question"], TOP_K)
    print(f"  📦 ChromaDB → {n} chunks")
    return {**etat, "resultat_vectoriel": contexte or "Aucun extrait technique trouvé."}

def noeud_skip_vector(etat: EtatDiagnostic) -> EtatDiagnostic:
    """Bypasse ChromaDB quand inutile (stats, véhicule, alertes)."""
    print("  ⏭️  ChromaDB ignoré (non nécessaire pour ce type)")
    return {**etat, "resultat_vectoriel": ""}


# ──────────────────────────────────────────────────────────────────
# Condition : faut-il appeler ChromaDB ?
# ──────────────────────────────────────────────────────────────────
def condition_vector(etat: EtatDiagnostic) -> Literal["vector", "skip_vector"]:
    return "vector" if etat["besoin_vector"] else "skip_vector"


# ──────────────────────────────────────────────────────────────────
# Nœud 4 — Analyser (construction du prompt)
# ──────────────────────────────────────────────────────────────────
def noeud_analyser(etat: EtatDiagnostic) -> EtatDiagnostic:
    resultat = analyser(etat)
    return {**etat, "prompt_utilisateur": resultat["prompt_utilisateur"]}


# ──────────────────────────────────────────────────────────────────
# Nœud 5 — LLM (Groq)
# ──────────────────────────────────────────────────────────────────
def noeud_llm(etat: EtatDiagnostic) -> EtatDiagnostic:
    completion = groq_client.chat.completions.create(
        model=MODELE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": etat["prompt_utilisateur"]}
        ],
        temperature=0.0,
        max_completion_tokens=2048,
        top_p=0.95,
        reasoning_effort="default",
        stream=True,
        stop=None
    )

    reponse = ""
    for chunk in completion:
        content = chunk.choices[0].delta.content
        if content:
            reponse += content

    print(f"  🤖 LLM → {len(reponse)} caractères générés")
    return {**etat, "reponse_llm": reponse.strip()}


# ──────────────────────────────────────────────────────────────────
# Construction du graphe V4
# ──────────────────────────────────────────────────────────────────
def construire_graphe() -> StateGraph:
    graphe = StateGraph(EtatDiagnostic)

    graphe.add_node("router",      noeud_router)
    graphe.add_node("sql",         noeud_sql)
    graphe.add_node("vector",      noeud_vector)
    graphe.add_node("skip_vector", noeud_skip_vector)
    graphe.add_node("analyser",    noeud_analyser)
    graphe.add_node("llm",         noeud_llm)

    graphe.set_entry_point("router")
    graphe.add_edge("router", "sql")

    # ── فرع شرطي : هل نحتاج ChromaDB؟ ──────────────────────────
    graphe.add_conditional_edges(
        "sql",
        condition_vector,
        {
            "vector":      "vector",
            "skip_vector": "skip_vector"
        }
    )

    graphe.add_edge("vector",      "analyser")
    graphe.add_edge("skip_vector", "analyser")
    graphe.add_edge("analyser",    "llm")
    graphe.add_edge("llm",          END)

    return graphe.compile()


agent_truck = construire_graphe()
print("✅ LangGraph Agent V4 compilé")
print("   router → sql → [vector | skip_vector] → analyser → llm")


# ── Interface principale ──────────────────────────────────────────
def poser_question(question: str) -> str:
    etat_initial: EtatDiagnostic = {
        "question":           question,
        "type_requete":       "",
        "besoin_vector":      False,
        "resultat_sql":       "",
        "resultat_vectoriel": "",
        "prompt_utilisateur": "",
        "reponse_llm":        ""
    }
    etat_final = agent_truck.invoke(etat_initial)
    return etat_final["reponse_llm"]


# ── Tests ─────────────────────────────────────────────────────────
tests = [
    ("DTC",      "Le code P0102 apparaît. Que faut-il vérifier ?"),
    ("Stats",    "Quel est le taux d'anomalies détectées ?"),
    ("Véhicule", "Quel est l'état du véhicule 42 ?"),
    ("Seuils",   "Quels sont les seuils de pression des pneus ?"),
    ("Alertes",  "Quels véhicules ont des alertes rouge actives ?"),
]

for label, question in tests:
    print(f"\n{'='*60}")
    print(f"TEST — {label} : {question}")
    rep = poser_question(question)
    print(rep[:400], "..." if len(rep) > 400 else "")

✅ LangGraph Agent V4 compilé
   router → sql → [vector | skip_vector] → analyser → llm

TEST — DTC : Le code P0102 apparaît. Que faut-il vérifier ?
  🔀 Router V4 → type: 'dtc' | vector: True
  🔎 Stratégie A — DTC : ['P0102']
  ✅ Stratégies utilisées : ['A']
  🗄️  SQL → 1 résultats
  📦 ChromaDB → 9 chunks
  🤖 LLM → 3725 caractères générés
<think>
Okay, let's tackle this query about the P0102 code. First, I need to recall what the code means. From the provided data, P0102 is related to the Mass Air Flow (MAF) sensor circuit input being low. The system in question is the intake system, and the component is the MAF sensor. The severity is medium.

Now, the user is asking what needs to be checked. I should start by referencing the DTC  ...

TEST — Stats : Quel est le taux d'anomalies détectées ?
  🔀 Router V4 → type: 'stats' | vector: False
  📊 Stratégie B — Stats globales
  ✅ Stratégies utilisées : ['B']
  🗄️  SQL → 7 résultats
  ⏭️  ChromaDB ignoré (non nécessaire pour ce type)
  🤖 LLM → 

In [7]:
# ═══════════════════════════════════════════════════════════════
# 📊  CELLULE 7 — Pipeline d'Évaluation (V4)
# ═══════════════════════════════════════════════════════════════

import json
import time


def charger_qa_dataset(chemin: str) -> list:
    if not os.path.exists(chemin):
        raise FileNotFoundError(f"Fichier QA introuvable : {chemin}")
    with open(chemin, "r", encoding="utf-8") as f:
        data = json.load(f)
    qas = data.get("questions_answers", [])
    print(f"✅ QA Dataset chargé : {len(qas)} questions")
    return qas


def generer_reponses_llm(qas: list, delai: float = 3.0,
                          chemin_sauvegarde: str = None) -> list:
    """
    Appelle poser_question() pour chaque Q&A.
    Recrée le fichier depuis zéro à chaque exécution.
    """
    resultats = []
    total     = len(qas)

    for idx, qa in enumerate(qas, start=1):
        question   = qa.get("question",   "")
        answer_ref = qa.get("answer",     "")
        difficulty = qa.get("difficulty", "facile")
        source     = qa.get("source_file", "")

        print(f"\n[{idx:02d}/{total}] 🔍 {question[:80]}...")

        reponse_llm  = ""
        type_requete = ""
        erreur       = None

        for tentative in range(2):
            try:
                etat_initial = {
                    "question":           question,
                    "type_requete":       "",
                    "besoin_vector":      False,
                    "resultat_sql":       "",
                    "resultat_vectoriel": "",
                    "prompt_utilisateur": "",
                    "reponse_llm":        ""
                }
                etat_final   = agent_truck.invoke(etat_initial)
                reponse_llm  = etat_final["reponse_llm"]
                type_requete = etat_final["type_requete"]
                erreur       = None
                break

            except Exception as e:
                erreur = str(e)
                print(f"  ⚠️  Tentative {tentative+1} échouée : {e}")
                if tentative == 0:
                    time.sleep(5)

        if erreur:
            reponse_llm  = f"ERREUR: {erreur}"
            type_requete = "erreur"
            print(f"  ❌ Échec définitif : {erreur}")

        resultats.append({
            "id":               idx,
            "source_file":      source,
            "question":         question,
            "difficulty":       difficulty,
            "type_requete":     type_requete,
            "reponse_correcte": answer_ref,
            "reponse_llm":      reponse_llm,
        })

        # ── Sauvegarde progressive ────────────────────────────────
        if chemin_sauvegarde:
            _sauvegarder(resultats, chemin_sauvegarde)
            print(f"  💾 Sauvegardé ({idx}/{total})")

        if idx < total:
            time.sleep(delai)

    print(f"\n✅ {len(resultats)} réponses générées.")
    return resultats


def _sauvegarder(resultats: list, chemin: str):
    """Sauvegarde les résultats — écrase le fichier existant."""
    os.makedirs(os.path.dirname(chemin), exist_ok=True)
    with open(chemin, "w", encoding="utf-8") as f:
        json.dump(
            {"dataset_name": "truck_llm_results", "results": resultats},
            f, ensure_ascii=False, indent=2
        )

def sauvegarder_resultats(resultats: list, chemin: str):
    _sauvegarder(resultats, chemin)
    print(f"✅ questionResults.json sauvegardé → {chemin}")


# ── Exécution ─────────────────────────────────────────────────────
qas       = charger_qa_dataset(QA_INPUT_FILE)
resultats = generer_reponses_llm(
    qas,
    delai=15.0,
    chemin_sauvegarde=OUTPUT_FILE
)
sauvegarder_resultats(resultats, OUTPUT_FILE)

✅ QA Dataset chargé : 49 questions

[01/49] 🔍 À quoi sert la zone verte du tachymètre dans le camion Volvo FH/FM ?...
  🔀 Router V4 → type: 'technique' | vector: True
  🔄 Stratégie F — Fallback maintenance général
  ✅ Stratégies utilisées : ['F']
  🗄️  SQL → 9 résultats
  📦 ChromaDB → 9 chunks
  🤖 LLM → 2665 caractères générés
  💾 Sauvegardé (1/49)

[02/49] 🔍 Pourquoi faut-il éviter la zone rouge du tachymètre ?...
  🔀 Router V4 → type: 'alertes' | vector: False
  🚨 Stratégie E — Alertes actives
  ✅ Stratégies utilisées : ['E']
  🗄️  SQL → 9 résultats
  ⏭️  ChromaDB ignoré (non nécessaire pour ce type)
  🤖 LLM → 3182 caractères générés
  💾 Sauvegardé (2/49)

[03/49] 🔍 Comment conduire de manière économique selon l'indicateur de pression du turboco...
  🔀 Router V4 → type: 'technique' | vector: True
  ⚙️  Stratégie D — Seuils thresholds
  ✅ Stratégies utilisées : ['D']
  🗄️  SQL → 5 résultats
  📦 ChromaDB → 9 chunks
  🤖 LLM → 3269 caractères générés
  💾 Sauvegardé (3/49)

[04/49] 🔍 Pour

In [8]:
# ═══════════════════════════════════════════════════════════════
# 📐  CELLULE 8 — Similarité Cosinus (V4)
# ═══════════════════════════════════════════════════════════════

import numpy as np
import re
from collections import defaultdict


def similarite_cosinus(vect_a: list, vect_b: list) -> float:
    """Calcule la similarité cosinus entre deux vecteurs numpy."""
    a = np.array(vect_a)
    b = np.array(vect_b)
    norme_a = np.linalg.norm(a)
    norme_b = np.linalg.norm(b)
    if norme_a == 0 or norme_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norme_a * norme_b))


def extraire_reponse_llm(texte: str) -> str:
    """Extrait la réponse après </think>. Retourne le texte tel quel sinon."""
    match = re.search(r'</think>\s*(.*)', texte, re.DOTALL)
    if match:
        return match.group(1).strip()
    return texte.strip()


def calculer_similarites(resultats: list, modele_emb=None) -> list:
    if modele_emb is None:
        if pipeline["ready"]:
            modele_emb = pipeline["embed_obj"]
        else:
            modele_emb = _charger_modele_embedding()

    resultats_enrichis = []
    n = len(resultats)
    print(f"🧮 Calcul de la similarité cosinus pour {n} paires ...")

    for i, res in enumerate(resultats, start=1):
        ref      = res.get("reponse_correcte", "")
        llm_brut = res.get("reponse_llm", "")
        llm      = extraire_reponse_llm(llm_brut)

        if ref.strip() and llm.strip():
            vec_ref = modele_emb.embed_query(ref)
            vec_llm = modele_emb.embed_query(llm)
            sim     = similarite_cosinus(vec_ref, vec_llm)
        else:
            sim = 0.0

        enrichi = res.copy()
        enrichi["cosine_similarity"] = round(sim, 4)
        resultats_enrichis.append(enrichi)

        print(f"  [{i:02d}/{n}] cosine = {sim:.4f} | "
              f"type: {res.get('type_requete','?'):10s} | "
              f"Q: {res['question'][:50]}...")

    # ── Statistiques globales ─────────────────────────────────────
    scores = [r["cosine_similarity"] for r in resultats_enrichis]
    print(f"\n📊 Statistiques globales ({n} questions) :")
    print(f"   Moyenne  : {np.mean(scores):.4f}")
    print(f"   Médiane  : {np.median(scores):.4f}")
    print(f"   Min      : {np.min(scores):.4f}")
    print(f"   Max      : {np.max(scores):.4f}")

    # ── ✅ V4 : Statistiques par type_requete ─────────────────────
    stats_par_type = defaultdict(list)
    for r in resultats_enrichis:
        type_req = r.get("type_requete", "inconnu")
        stats_par_type[type_req].append(r["cosine_similarity"])

    print(f"\n📊 Similarité moyenne par type de requête :")
    print(f"   {'Type':<15} {'Nb':>4}   {'Moyenne':>8}   {'Min':>8}   {'Max':>8}")
    print(f"   {'-'*50}")
    for type_req, scores_type in sorted(stats_par_type.items()):
        print(f"   {type_req:<15} {len(scores_type):>4}   "
              f"{np.mean(scores_type):>8.4f}   "
              f"{np.min(scores_type):>8.4f}   "
              f"{np.max(scores_type):>8.4f}")

    # ── ✅ V4 : Statistiques par difficulté ───────────────────────
    stats_par_diff = defaultdict(list)
    for r in resultats_enrichis:
        diff = r.get("difficulty", "inconnu")
        stats_par_diff[diff].append(r["cosine_similarity"])

    print(f"\n📊 Similarité moyenne par difficulté :")
    print(f"   {'Difficulté':<15} {'Nb':>4}   {'Moyenne':>8}")
    print(f"   {'-'*35}")
    for diff, scores_diff in sorted(stats_par_diff.items()):
        print(f"   {diff:<15} {len(scores_diff):>4}   "
              f"{np.mean(scores_diff):>8.4f}")

    return resultats_enrichis


# ── Exécution ─────────────────────────────────────────────────────
resultats_avec_cosinus = calculer_similarites(resultats)
sauvegarder_resultats(resultats_avec_cosinus, OUTPUT_FILE)
print("✅ questionResults.json mis à jour avec cosine_similarity.")

🧮 Calcul de la similarité cosinus pour 49 paires ...
  [01/49] cosine = 0.8035 | type: technique  | Q: À quoi sert la zone verte du tachymètre dans le ca...
  [02/49] cosine = 0.7121 | type: alertes    | Q: Pourquoi faut-il éviter la zone rouge du tachymètr...
  [03/49] cosine = 0.6653 | type: technique  | Q: Comment conduire de manière économique selon l'ind...
  [04/49] cosine = 0.9266 | type: technique  | Q: Pourquoi Volvo recommande-t-il une conduite pruden...
  [05/49] cosine = 0.8825 | type: général    | Q: Que peut provoquer l'utilisation d'un carburant di...
  [06/49] cosine = 0.5652 | type: technique  | Q: Le voyant de pression d'huile s'allume pendant la ...
  [07/49] cosine = 0.7047 | type: technique  | Q: Quel est le rôle du compte-tours dans le camion Vo...
  [08/49] cosine = 0.7534 | type: général    | Q: Pourquoi est-il déconseillé de rouler à haute vite...
  [09/49] cosine = 0.6847 | type: technique  | Q: Quel comportement est recommandé pendant les premi...
  [10/49] c

In [9]:
# ═══════════════════════════════════════════════════════════════
# 🏆  CELLULE 9 — Système de Scoring Multi-Modèles (V4)
# ═══════════════════════════════════════════════════════════════
#
#  Pour chaque question :
#    1. Score Cosinus  = cosine_similarity × 100        (→ /100)
#    2. Score Gemini   = (note/5) × 100                 (→ /100)
#    3. Score Claude   = (note/5) × 100                 (→ /100)
#    4. Score DeepSeek = (note/5) × 100                 (→ /100)
#
#  Score final par question = (Cosinus + Gemini + Claude + DeepSeek) / 4
#  Score global /100 = Σ(score_final_i × coeff_i) / Σ(coeff_i)
# ═══════════════════════════════════════════════════════════════

def _saisir_note(label: str) -> float:
    """Saisie d'une note entre 0 et 5 avec validation."""
    while True:
        try:
            note = float(input(f"  Note {label} (0-5) : "))
            if 0.0 <= note <= 5.0:
                return note
            print("  ⚠️  Valeur invalide — entrez un nombre entre 0 et 5.")
        except ValueError:
            print("  ⚠️  Nombre invalide.")


def _score_evaluateur_sur_100(note: float) -> float:
    """Convertit une note /5 en score /100."""
    return round((note / 5.0) * 100.0, 4)


def saisir_notes_et_calculer(
    resultats: list,
    modeles: list = None
) -> list:
    """
    Pour chaque question :
      - Affiche : question, réponse correcte, réponse LLM, similarité cosinus
      - Demande une note /5 pour chaque modèle évaluateur
      - Calcule le score final = moyenne(Cosinus, Gemini, Claude, DeepSeek)
    """
    if modeles is None:
        modeles = MODELES_EVALUATION   # ["Gemini", "Claude", "DeepSeek"]

    resultats_enrichis = []
    total = len(resultats)

    print("=" * 72)
    print("🏆  SAISIE DES NOTES ÉVALUATEURS")
    print(f"    Modèles : Cosinus (auto) + {' + '.join(modeles)}")
    print(f"    Moyenne des {1 + len(modeles)} scores → Score final (/100)")
    print("=" * 72)

    for res in resultats:
        idx        = res["id"]
        question   = res["question"]
        difficulty = res.get("difficulty", "facile")
        coeff      = COEFF_DIFFICULTE.get(difficulty, 1.0)
        reponse    = res.get("reponse_llm", "")
        cosine     = res.get("cosine_similarity", 0.0)
        # ✅ V4 : affichage du type_requete
        type_req   = res.get("type_requete", "?")

        print(f"\n{'─' * 72}")
        print(f"  Question [{idx:02d}/{total}]  |  "
              f"Type: {type_req:10}  |  "
              f"Difficulté: {difficulty.upper():8}  |  "
              f"Coeff: ×{coeff}")
        print(f"{'─' * 72}")
        print(f"  ❓ {question}")
        print(f"  ✅ Réponse correcte : {res['reponse_correcte']}")
        print(f"  🤖 Réponse LLM      : "
              f"{reponse[:250]}{'...' if len(reponse) > 250 else ''}")
        print()

        # ── Score 1 : Cosinus (automatique) ──────────────────────
        cosinus_score_100 = round(cosine * 100.0, 4)
        print(f"  📐 Score Cosinus : {cosine:.4f}  →  {cosinus_score_100:.2f}/100")
        print()

        # ── Scores 2-4 : Évaluateurs ─────────────────────────────
        evaluations = {
            "Cosinus": {"score_sur_100": cosinus_score_100}
        }
        scores_evaluateurs = [cosinus_score_100]

        for modele in modeles:
            note      = _saisir_note(modele)
            score_100 = _score_evaluateur_sur_100(note)
            evaluations[modele] = {
                "note_sur_5":    note,
                "score_sur_100": score_100
            }
            scores_evaluateurs.append(score_100)
            print(f"    → {modele:<10} : {note}/5  →  {score_100:.2f}/100")

        # ── Score final ───────────────────────────────────────────
        score_final = round(
            sum(scores_evaluateurs) / len(scores_evaluateurs), 4
        )
        print()
        print(f"  ⭐ Score final [{idx:02d}] = "
              f"({' + '.join(f'{s:.2f}' for s in scores_evaluateurs)}) "
              f"/ {len(scores_evaluateurs)} = {score_final:.2f}/100")

        enrichi = res.copy()
        enrichi["evaluations"]          = evaluations
        enrichi["score_final_question"] = score_final
        resultats_enrichis.append(enrichi)

    print(f"\n{'═' * 72}")
    print(f"✅ {len(resultats_enrichis)} questions évaluées.")
    return resultats_enrichis


def calculer_score_global(resultats: list) -> dict:
    """
    Score global /100 par évaluateur + score final pondéré par difficulté.
    """
    modeles_tous         = ["Cosinus"] + MODELES_EVALUATION
    somme_ponderee       = {m: 0.0 for m in modeles_tous}
    somme_ponderee_final = 0.0
    total_coeff          = 0.0

    for res in resultats:
        coeff = COEFF_DIFFICULTE.get(res.get("difficulty", "facile"), 1.0)
        evals = res.get("evaluations", {})
        s_fin = res.get("score_final_question", 0.0)

        somme_ponderee_final += s_fin * coeff
        total_coeff          += coeff

        for m in modeles_tous:
            if m in evals:
                s = evals[m].get("score_sur_100", 0.0)
                somme_ponderee[m] += s * coeff

    scores = {
        m: round(somme_ponderee[m] / total_coeff, 2) if total_coeff > 0 else 0.0
        for m in modeles_tous
    }
    scores["SCORE_FINAL_GLOBAL"] = (
        round(somme_ponderee_final / total_coeff, 2) if total_coeff > 0 else 0.0
    )
    return scores


# ── Exécution ─────────────────────────────────────────────────────
resultats_notes = saisir_notes_et_calculer(resultats_avec_cosinus)
sauvegarder_resultats(resultats_notes, OUTPUT_FILE)
print("✅ Évaluations sauvegardées dans questionResults.json.")

🏆  SAISIE DES NOTES ÉVALUATEURS
    Modèles : Cosinus (auto) + Gemini + claude + DeepSeek
    Moyenne des 4 scores → Score final (/100)

────────────────────────────────────────────────────────────────────────
  Question [01/49]  |  Type: technique   |  Difficulté: FACILE    |  Coeff: ×1.0
────────────────────────────────────────────────────────────────────────
  ❓ À quoi sert la zone verte du tachymètre dans le camion Volvo FH/FM ?
  ✅ Réponse correcte : La zone verte du tachymètre est destinée à la conduite normale et permet un fonctionnement approprié du moteur.
  🤖 Réponse LLM      : <think>
Okay, let's tackle this question. The user is asking about the purpose of the green zone on the tachometer in a Volvo FH/FM truck. First, I need to recall the information provided in the context.

Looking at the EXTRAITS_MANUEL section, there...

  📐 Score Cosinus : 0.8035  →  80.35/100

    → Gemini     : 5.0/5  →  100.00/100
    → claude     : 5.0/5  →  100.00/100
    → DeepSeek   : 5.0/5  → 

In [10]:
# ═══════════════════════════════════════════════════════════════
# 📋  CELLULE 10 — Rapport Final & Scores /100
# ═══════════════════════════════════════════════════════════════

import numpy as np


def afficher_rapport_final(resultats: list):
    """
    Affiche :
    - Scores globaux /100 par modèle évaluateur
    - Statistiques cosinus par difficulté
    - Tableau récapitulatif par question
    """
    modeles = MODELES_EVALUATION

    # ── Scores globaux /100 ───────────────────────────────────────
    scores_globaux = calculer_score_global(resultats)  # ✅ معامل واحد فقط

    print("\n" + "═" * 70)
    print("🏆  SCORES GLOBAUX /100  (Σ note × coeff / Σ 5 × coeff × 100)")
    print("═" * 70)

    for modele, score in scores_globaux.items():
        barre = "█" * int(score / 5)
        print(f"  {modele:<20} : {score:6.2f}/100  {barre}")

    # ── Cosinus moyen par difficulté ──────────────────────────────
    print("\n" + "─" * 70)
    print("📐  SIMILARITÉ COSINUS par niveau de difficulté")
    print("─" * 70)

    for diff in ["facile", "moyen", "difficile"]:
        valeurs = [
            r["cosine_similarity"]
            for r in resultats
            if r.get("difficulty") == diff and "cosine_similarity" in r
        ]
        if valeurs:
            print(f"  {diff.capitalize():<12} : {np.mean(valeurs):.4f} "
                  f"(n={len(valeurs)} | min={min(valeurs):.3f} max={max(valeurs):.3f})")

    # ── Tableau par question ──────────────────────────────────────
    print("\n" + "─" * 70)
    print("📊  DÉTAIL PAR QUESTION")
    print("─" * 70)
    header = f"{'ID':>3}  {'Diff':<10}  {'Cosine':>7}  "
    for m in modeles:
        header += f"{m:>10}  "
    header += "{'Final':>8}  Question"
    print(header)
    print("─" * 70)

    for res in resultats:
        idx    = res["id"]
        diff   = res.get("difficulty", "?")[:6]
        cosine = res.get("cosine_similarity", 0.0)
        evals  = res.get("evaluations", {})
        score_final = res.get("score_final_question", 0.0)
        q_short = res["question"][:40] + "..."

        ligne = f"{idx:>3}  {diff:<10}  {cosine:>7.4f}  "
        for m in modeles:
            note = evals.get(m, {}).get("note_sur_5", "-")
            note_str = f"{note:.1f}/5" if isinstance(note, float) else str(note)
            ligne += f"{note_str:>10}  "
        ligne += f"{score_final:>7.2f}  {q_short}"
        print(ligne)

    # ── Sauvegarde du rapport ─────────────────────────────────────
    rapport = {
        "scores_globaux": scores_globaux,
        "statistiques_cosinus": {
            d: round(
                np.mean([r["cosine_similarity"] for r in resultats
                         if r.get("difficulty") == d and "cosine_similarity" in r]),
                4
            )
            for d in ["facile", "moyen", "difficile"]
            if any(r.get("difficulty") == d and "cosine_similarity" in r for r in resultats)
        },
        "total_questions": len(resultats),
        "coefficients": COEFF_DIFFICULTE,
        "modeles_evalues": modeles,
        "details": resultats
    }

    rapport_path = os.path.join(EVAL_DIR, "rapport_final.json")
    os.makedirs(os.path.dirname(rapport_path), exist_ok=True)
    with open(rapport_path, "w", encoding="utf-8") as f:
        json.dump(rapport, f, ensure_ascii=False, indent=2)

    # Mise à jour questionResults.json avec les évaluations finales
    sauvegarder_resultats(resultats, OUTPUT_FILE)

    print(f"\n✅ Rapport final sauvegardé → {rapport_path}")
    print(f"✅ questionResults.json mis à jour → {OUTPUT_FILE}")


# ── Affichage du rapport ──────────────────────────────────────────
afficher_rapport_final(resultats_notes)


══════════════════════════════════════════════════════════════════════
🏆  SCORES GLOBAUX /100  (Σ note × coeff / Σ 5 × coeff × 100)
══════════════════════════════════════════════════════════════════════
  Cosinus              :  71.01/100  ██████████████
  Gemini               : 100.00/100  ████████████████████
  claude               :  97.14/100  ███████████████████
  DeepSeek             :  95.43/100  ███████████████████
  SCORE_FINAL_GLOBAL   :  90.90/100  ██████████████████

──────────────────────────────────────────────────────────────────────
📐  SIMILARITÉ COSINUS par niveau de difficulté
──────────────────────────────────────────────────────────────────────
  Facile       : 0.7069 (n=17 | min=0.543 max=0.927)
  Moyen        : 0.7413 (n=22 | min=0.580 max=0.882)
  Difficile    : 0.6615 (n=10 | min=0.509 max=0.740)

──────────────────────────────────────────────────────────────────────
📊  DÉTAIL PAR QUESTION
──────────────────────────────────────────────────────────────────────
 